In [17]:
import os
os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"
SAVE = True
PARALLEL = True   # only takes effect when SAVE=True (see bottom of file)

import re
import glob
from datetime import datetime, timedelta
from io import BytesIO

import numpy as np
from PIL import Image

import matplotlib
if SAVE:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt

import earthkit.data as ekd
import earthkit.plots as ekp


# ==========================================================
# SETTINGS
# ==========================================================

START_TIME = datetime(2026, 1, 11, 00)
END_TIME   = datetime(2026, 1, 18, 23)

RUN_SUFFIX = "638"
MEMBER = "000"

ICON_BASE = f"/store_new/mch/msopr/osm/ICON-CH1-EPS/FCST{START_TIME.year % 100:02d}"

OUTPUT_DIR = os.path.expanduser("~/SatICON")
SAT_DIR = os.path.join(OUTPUT_DIR, "satellite_raw")
SAT_NAME_FMT = "MSG_RGB-DayNightFog_cosmo1_{:%Y%m%d%H%M}.png"


# cloud cover
CLCT_LEVELS = list(range(0, 101, 10))
CLCT_CMAP = "Blues"

# difference
DIFF_LEVELS = list(range(-100, 110, 10))
DIFF_CMAP = "RdBu_r"

# big number annotation on the difference plots
MEAN_LABEL_FONTSIZE = 22


PLOT_DPI = 75
SAVE_DPI = 90

FIGSIZE = (24, 12)

# only used if PARALLEL = True and SAVE = True
MAX_WORKERS = 8


# ==========================================================
# FIND ICON FILE FOR ANY LEAD TIME
# ==========================================================

def find_icon_file(run_time, lead_hours):
    icon_dir = os.path.join(ICON_BASE, f"{run_time:%y%m%d%H}_{RUN_SUFFIX}", "grib")
    days, hours = divmod(lead_hours, 24)
    fname = f"i1eff{days:02d}{hours:02d}0000_{MEMBER}"
    path = os.path.join(icon_dir, fname)
    return path if os.path.exists(path) else None

def find_sat_file(valid_time):
    sat_file = os.path.join(SAT_DIR,SAT_NAME_FMT.format(valid_time))
    return sat_file if os.path.exists(sat_file) else None

# ==========================================================
# READ CLCT
# ==========================================================

def read_clct(grib_file):

    if not os.path.exists(grib_file):
        raise FileNotFoundError(grib_file)

    data = ekd.from_source(
        "file",
        grib_file
    ).to_fieldlist()

    if len(data) == 0:
        raise RuntimeError(
            f"No fields found in {grib_file}"
        )

    try:
        CLCT = data.sel(
            {
                "parameter.variable": "CLCL"
            }
        )

    except Exception:
        raise RuntimeError(
            f"CLCT not found in {grib_file}"
        )

    return CLCT

# ==========================================================
# DOMAIN MEAN OF A FIELD
# ==========================================================

def _first_field(obj):
    try:
        return obj[0]
    except Exception:
        return obj


def domain_mean(field_or_list):

    f = _first_field(field_or_list)

    values = np.asarray(f.to_numpy(flatten=True), dtype=float)
    values = np.where(np.abs(values) > 100, np.nan, values)

    return float(np.nanmean(values))

# ==========================================================
# RENDER CLCT FIELD
# ==========================================================

def render_field(field, title, difference=False, mean_value=None):

    chart = ekp.Map()

    if difference:

        chart.contourf(
            field,
            levels=DIFF_LEVELS,
            cmap=DIFF_CMAP,
            extend="both"
        )

    else:

        chart.contourf(
            field,
            levels=CLCT_LEVELS,
            cmap=CLCT_CMAP,
            extend="neither"
        )

    chart.coastlines()
    chart.borders()
    chart.gridlines()

    if difference and mean_value is not None:
        chart.title(f"{title} Mean Δ = {mean_value:+.1f} pp")
    else:
        chart.title(title)

    chart.legend(
        label="%",
        shrink=0.8,
        pad=0.02
    )

    buffer = BytesIO()

    chart.fig.savefig(
        buffer,
        format="png",
        dpi=PLOT_DPI,
        bbox_inches=None
    )

    buffer.seek(0)

    img = Image.open(buffer).convert("RGB")

    plt.close(chart.fig)

    return img

# ==========================================================
# IMAGE HELPERS
# ==========================================================

def crop_white(img):

    bbox = (
        img.convert("L")
        .point(lambda p: 0 if p > 245 else 255)
        .getbbox()
    )

    return img.crop(bbox) if bbox else img



def make_taller(img, factor=1.20):

    return img.resize(
        (
            img.width,
            int(img.height * factor)
        ),
        Image.Resampling.LANCZOS
    )

# ==========================================================
# DRAW FRAME
# ==========================================================

def draw_frame(
    sat_file,
    valid_time,
    icon1,
    icon25,
    icon7,
    icon31
):

    sat_img = Image.open(
        sat_file
    ).convert("RGB")

    # read fields
    clct1 = read_clct(icon1)
    clct25 = read_clct(icon25)

    clct7 = read_clct(icon7)
    clct31 = read_clct(icon31)

    # differences
    diff1 = clct1 - clct25
    diff7 = clct7 - clct31

    # mean stats for the annotation
    mean1 = domain_mean(diff1)
    mean7 = domain_mean(diff7)

    # render images

    img1 = render_field(
        clct1,
        f"ICON current run +1h\n{valid_time:%Y-%m-%d %H:%M UTC}"
    )

    img25 = render_field(
        clct25,
        f"ICON previous run +25h\n{valid_time:%Y-%m-%d %H:%M UTC}"
    )

    imgdiff1 = render_field(
        diff1,
        "Difference\n(+1h - +25h)",
        difference=True,
        mean_value=mean1
    )

    img7 = render_field(
        clct7,
        f"ICON current run +7h\n{valid_time:%Y-%m-%d %H:%M UTC}"
    )

    img31 = render_field(
        clct31,
        f"ICON previous run +31h\n{valid_time:%Y-%m-%d %H:%M UTC}"
    )

    imgdiff7 = render_field(
        diff7,
        "Difference\n(+7h - +31h)",
        difference=True,
        mean_value=mean7
    )

    images = [
        img1,
        img25,
        imgdiff1,
        img7,
        img31,
        imgdiff7
    ]

    images = [
        make_taller(crop_white(i))
        for i in images
    ]


    fig = plt.figure(
        figsize=FIGSIZE
    )

    # satellite
    ax_sat = fig.add_axes(
        [
            0.01,
            0.18,
            0.22,
            0.65
        ]
    )

    ax_sat.imshow(
        sat_img
    )

    ax_sat.set_title(
        f"MSG RGB Satellite\n{valid_time:%Y-%m-%d %H:%M UTC}"
    )

    ax_sat.axis("off")

    positions = [

        # top row
        [0.24, 0.55, 0.23, 0.35],
        [0.50, 0.55, 0.23, 0.35],
        [0.76, 0.55, 0.23, 0.35],

        # bottom row
        [0.24, 0.10, 0.23, 0.35],
        [0.50, 0.10, 0.23, 0.35],
        [0.76, 0.10, 0.23, 0.35],

    ]

    for img, pos in zip(images, positions):

        ax = fig.add_axes(pos)
        ax.imshow(
            img,
            aspect="auto"
        )
        ax.axis("off")

    if SAVE:

        out_path = os.path.join(
            OUTPUT_DIR,
            f"SatICON_compare_{valid_time:%Y%m%d%H}.png"
        )

        fig.savefig(
            out_path,
            dpi=SAVE_DPI,
            bbox_inches="tight"
        )

        print(
            "Saved:",
            out_path
        )

    else:

        plt.show()

    plt.close(fig)

# ==========================================================
# COLLECT THE LIST OF FRAMES TO PROCESS
# ==========================================================

def collect_frames():

    frames = []
    n_skipped = 0

    run_time = START_TIME

    while run_time.hour % 3 != 0:
        run_time += timedelta(hours=1)

    while run_time <= END_TIME:

        today_run = run_time
        yesterday_run = run_time - timedelta(hours=24)

        valid1 = today_run + timedelta(hours=1)
        valid7 = today_run + timedelta(hours=7)

        sat_file = find_sat_file(valid1)

        if sat_file is None:
            n_skipped += 1
            run_time += timedelta(hours=3)
            continue

        icon1 = find_icon_file(today_run, 1)
        icon25 = find_icon_file(yesterday_run, 25)
        icon7 = find_icon_file(today_run, 7)
        icon31 = find_icon_file(yesterday_run, 31)

        files = [icon1, icon25, icon7, icon31]

        if any(f is None for f in files):
            n_skipped += 1
            run_time += timedelta(hours=3)
            continue

        frames.append((sat_file, valid1, icon1, icon25, icon7, icon31))

        run_time += timedelta(hours=3)

    return frames, n_skipped


def _draw_frame_unpack(args):
    draw_frame(*args)

# ==========================================================
# MAIN
# ==========================================================

if __name__ == "__main__":

    if SAVE:
        os.makedirs(OUTPUT_DIR, exist_ok=True)

    frames, n_skipped = collect_frames()

    if PARALLEL and SAVE:
        import concurrent.futures
        with concurrent.futures.ProcessPoolExecutor(max_workers=MAX_WORKERS) as pool:
            list(pool.map(_draw_frame_unpack, frames))
        n_processed = len(frames)

    else:
        n_processed = 0
        for args in frames:
            draw_frame(*args)
            n_processed += 1
    print(
        f"\nDone: {n_processed} frame(s) plotted, {n_skipped} skipped."
    )

Saved: /users/jdelbeke/SatICON/SatICON_compare_2026011104.png
Saved: /users/jdelbeke/SatICON/SatICON_compare_2026011101.png
Saved: /users/jdelbeke/SatICON/SatICON_compare_2026011116.png
Saved: /users/jdelbeke/SatICON/SatICON_compare_2026011119.png
Saved: /users/jdelbeke/SatICON/SatICON_compare_2026011122.png
Saved: /users/jdelbeke/SatICON/SatICON_compare_2026011110.png
Saved: /users/jdelbeke/SatICON/SatICON_compare_2026011107.png
Saved: /users/jdelbeke/SatICON/SatICON_compare_2026011113.png
Saved: /users/jdelbeke/SatICON/SatICON_compare_2026011201.png
Saved: /users/jdelbeke/SatICON/SatICON_compare_2026011207.png
Saved: /users/jdelbeke/SatICON/SatICON_compare_2026011204.png
Saved: /users/jdelbeke/SatICON/SatICON_compare_2026011216.png
Saved: /users/jdelbeke/SatICON/SatICON_compare_2026011213.png
Saved: /users/jdelbeke/SatICON/SatICON_compare_2026011219.png
Saved: /users/jdelbeke/SatICON/SatICON_compare_2026011210.png
Saved: /users/jdelbeke/SatICON/SatICON_compare_2026011222.png
Saved: /